# Kernel ICL transience: paper figures

Loads a results pickle written by `rf_transience_closed_form.ipynb` (`rf_transience_closed_*.pkl` in
`RESULTPATH`) and plots the measured test error against $n_\mathrm{train}$ at a fixed task diversity $T$,
with the dMMSE and Ridge oracles as horizontal lines.

The curve is the mean over trials at the ID-validation-chosen ridge (`record["curve"]`), with a $\pm 1$
standard deviation band across trials (none with one trial); the oracles come from `record["oracles"]`.

Every trial draws its **own task pool**, and at small $T$ the pool sets the level of the whole curve, so
each trial is first shifted by $\Delta_i = \mathrm{dMMSE}_i - \mathrm{dMMSE}_\mathrm{baseline}$
(`ALIGN`). That puts every pool on the baseline pool's level and leaves the plotted dMMSE line exactly
on it. Averaging the raw curves instead mixes pools, which inflates the band and flattens the dip. Set `SIDES = ("ood", "id")` for OOD and ID side by side.
Several pickles can be overlaid through `RUNS`; the oracle lines then come from the first one.

## Colab setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os

GIT_USER = "JoeyTurn"
GIT_REPO = "kernels_can_icl"

!git clone https://github.com/{GIT_USER}/{GIT_REPO}.git

sys.path.insert(0, "/content/kernels_can_icl")
sys.path.insert(0, "/content/kernels_can_icl/expts/rf_scaling")

RESULTPATH = "/content/drive/MyDrive/results"
os.environ["RESULTPATH"] = RESULTPATH
os.makedirs(RESULTPATH, exist_ok=True)

!git clone https://github.com/dkarkada/expt-core.git

sys.path.insert(0, "/content/expt-core")
from misc import rcsetup
rcsetup()

In [ ]:
import os, sys, glob, pickle
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import ticker

if os.path.exists("/content/kernels_can_icl"):
    # Colab: re-add the paths in case the runtime was restarted after the setup cell
    for p in ("/content/kernels_can_icl", "/content/kernels_can_icl/expts/rf_scaling", "/content/expt-core"):
        if p not in sys.path:
            sys.path.insert(0, p)
    RESULTPATH = os.environ.get("RESULTPATH", "/content/drive/MyDrive/results")
else:
    ROOT = Path.cwd()
    while ROOT != ROOT.parent and not (ROOT / "kernels_can_icl" / "__init__.py").exists():
        ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT))
    sys.path.insert(0, str(ROOT / "expts" / "rf_scaling"))
    sys.path.insert(0, str(ROOT.parent / "expt-core"))
    RESULTPATH = str(ROOT / "expts" / "rf_scaling" / "results")

try:
    from misc import rcsetup
    rcsetup()
except ImportError:                     # the same settings as expt-core's misc.rcsetup
    plt.rc("figure", dpi=120, facecolor=(1, 1, 1))
    plt.rc("font", family="stixgeneral", size=12)
    plt.rc("axes", facecolor=(1, .99, .95), titlesize=12)
    plt.rc("mathtext", fontset="cm")
    plt.rc("pdf", fonttype=42)
    plt.rc("ps", fonttype=42)
print("RESULTPATH:", RESULTPATH)

## Settings

`RUNS` maps a legend label to a pickle path (absolute, or a file name inside `RESULTPATH`); `None`
picks the newest non-smoke `rf_transience_closed_*.pkl` there. One figure is drawn per T in `CURVE_TS`.

In [ ]:
RUNS       = {"RF + ridge": None}   # legend label -> pickle path; None = newest closed-form pickle
CURVE_TS   = [8]                    # one figure per T
SIDES      = ("ood",)               # ("ood", "id") for OOD | ID side by side
YSCALE     = "log"                  # as in fig 2; "linear" also works
YLIM       = {}                     # e.g. {"ood": (0.25, 1.5)}; default fits the curves and the oracles
SD_BAND    = True                   # +/- 1 standard deviation across trials (none with 1 trial)
ORA_BAND   = True                   # shade that spread on the oracle lines too; aligning on the dMMSE
                                    #   makes its own spread 0, so only Ridge shows one then
ALIGN      = "dmmse"                # put every trial on one pool's level before averaging: shift trial i
                                    #   by Delta_i = dMMSE_i - dMMSE_baseline (None = raw mean)
ALIGN_BASE = "first"                # the baseline pool: the "first" trial, or the "mean" over trials
LEGEND_LOC = "lower left"           # a lower legend sits just above an oracle line along the bottom
FONTSIZE   = 15                     # tick labels and legend; axis labels +2, the T label +6
T_LABEL    = True                   # the in-plot "T=..." label
TOP_FREE   = 0.25                   # keep this much of the axis clear under the T label and legend
XLABEL     = r"Training Samples $n$"

In [ ]:
def newest_pickle(resultpath, pattern):
    """Newest matching pickle, preferring full runs over _smoke ones."""
    paths = glob.glob(os.path.join(resultpath, pattern))
    if not paths:
        raise FileNotFoundError(f"no {pattern} in {resultpath}")
    full = [p for p in paths if "_smoke" not in os.path.basename(p)]
    return max(full or paths, key=os.path.getmtime)


def resolve(path, pattern):
    """A RUNS entry -> a file: None = the newest match in RESULTPATH, a bare name is looked up there."""
    path = path or newest_pickle(RESULTPATH, pattern)
    if not os.path.isabs(path) and not os.path.exists(path):
        path = os.path.join(RESULTPATH, path)
    return path


runs, metas = [], []
for label, path in RUNS.items():
    path = resolve(path, "rf_transience_closed_*.pkl")
    with open(path, "rb") as f:
        blob = pickle.load(f)
    meta, records = blob["meta"], blob["records"]
    runs.append((label, records))
    metas.append(meta)
    trials = {T: sum(r["T"] == T for r in records) for T in sorted({r["T"] for r in records})}
    ns = [c["n"] for r in records for c in r["curve"]]
    print(f"{label}: {os.path.basename(path)}")
    print(f"    trials per T {trials}   P={meta['P']:,}   width {meta['width_factor']}x median"
          f"   n {min(ns):,} .. {max(ns):,}")

META = metas[0]
TAG = f"d{META['config']['dim']}p{META['pairs']}{'_smoke' if META.get('smoke') else ''}"

## Figure

In [ ]:
# figure 2's palette, in series order
STYLES = [("#11968C", "o"), ("#6C63F0", "s"), ("#D97706", "^"), ("#C2185B", "D"), ("#2E7D32", "v")]
ORACLES = (("dmmse", "dMMSE Oracle", "#DC143C", 2.0), ("ridge", "Ridge Oracle", "black", 2.8))
SIDE_NAME = {"ood": "OOD", "id": "ID"}


def pool_offsets(records, T, side, align="dmmse", base="first"):
    """{trial: Delta_i} putting every trial's pool on one baseline level, and that baseline's dMMSE.

    Each trial draws its own task pool (the seed is keyed on T *and* the trial), and at small T the pool
    sets the level of the whole curve -- pools whose dMMSE differs by 0.1 shift their curves by about
    that much. Averaging the raw curves then inflates the band and flattens the dip. So everything a
    trial produces is shifted by Delta_i = dMMSE_i - dMMSE_baseline before it is averaged, which puts
    every pool on the baseline pool's level; the plotted dMMSE line lands exactly on that baseline.
    align=None turns this off (raw averaging). base: "first" trial's pool, or the "mean" over pools.
    """
    recs = sorted((r for r in records if r["T"] == T), key=lambda r: r.get("trial", 0))
    dm = [r["oracles"][f"dmmse_{side}"] for r in recs]
    if not dm:
        return {}, np.nan
    if not align:
        return {}, float(np.mean(dm))
    b = dm[0] if base == "first" else float(np.mean(dm))
    return {r.get("trial", i): d - b for i, (r, d) in enumerate(zip(recs, dm))}, b


def curve_stats(records, T, side, band=True, offs=None):
    """(n, mean, sd) of the measured curve at task diversity T, over trials, each shifted onto the
    baseline pool by `offs` (see pool_offsets). sd is the sample standard deviation across trials
    (never the standard error); 0 with one trial or band=False."""
    offs = offs or {}
    recs = [r for r in records if r["T"] == T and r.get("curve")]
    ns = sorted({c["n"] for r in recs for c in r["curve"]})
    m, s = [], []
    for n in ns:
        v = np.array([c[side] - offs.get(r.get("trial", 0), 0.0)
                      for r in recs for c in r["curve"] if c["n"] == n], float)
        v = v[np.isfinite(v)]
        m.append(v.mean() if v.size else np.nan)
        s.append(v.std(ddof=1) if band and v.size > 1 else 0.0)
    return np.array(ns, float), np.array(m), np.array(s)


def oracle_stats(records, T, key, side, offs=None, band=True):
    """(mean, sd) of an oracle ('dmmse' or 'ridge') over trials, on the same shifted pools as the
    curves. Note that aligning on the dMMSE makes every trial's dMMSE the baseline, so its sd is then
    exactly 0 -- the pool spread has moved into the curves, which are now read against their own pool."""
    offs = offs or {}
    v = np.array([r["oracles"][f"{key}_{side}"] - offs.get(r.get("trial", 0), 0.0)
                  for r in records if r["T"] == T], float)
    return float(v.mean()), float(v.std(ddof=1)) if band and v.size > 1 else 0.0


def theory_curve(records, T, side, offs=None):
    """(n, trial mean) of the theory curve record["pred"], shifted like the curves; None if absent."""
    offs = offs or {}
    recs = [r for r in records if r["T"] == T and r.get("pred")]
    if not recs:
        return None
    n = np.asarray(recs[0]["pred"]["n"], float)
    ys = [np.asarray(r["pred"][side], float) - offs.get(r.get("trial", 0), 0.0)
          for r in recs if len(r["pred"][side]) == len(n)]
    return n, np.mean(ys, 0)


def log_ticks(lo, hi, max_ticks=7):
    """Round tick values for a log axis: the finest of {1,1.2,1.5,2,3,4,5,6,8}, {1,2,3,5}, {1,2,5}, {1}
    per decade that puts at most max_ticks inside [lo, hi]."""
    decades = 10.0 ** np.arange(np.floor(np.log10(lo)), np.ceil(np.log10(hi)) + 1)
    for mant in ((1, 1.2, 1.5, 2, 3, 4, 5, 6, 8), (1, 2, 3, 5), (1, 2, 5), (1,)):
        vals = [float(f"{m * d:.6g}") for d in decades for m in mant]
        vals = [v for v in vals if lo <= v <= hi]
        if 2 <= len(vals) <= max_ticks:
            return vals
    return None


def default_ylim(ax, xys, yscale, top_free, x_from=0.38):
    """y-limits that fit everything drawn, raised if needed so the top `top_free` of the axis is
    clear of data right of x-fraction `x_from` -- where the T label and an upper-right legend sit.
    The x-limits must already be set (call after plotting)."""
    log = yscale == "log"
    ok = lambda y: np.isfinite(y) & (y > 0) if log else np.isfinite(y)
    ys = np.concatenate([y[ok(y)] for _, y in xys])
    if log:
        lo, hi = ys.min() / 1.1, ys.max() * 1.08
    else:
        pad = 0.06 * (ys.max() - ys.min())
        lo, hi = ys.min() - pad, ys.max() + pad
    x0, x1 = np.log10(ax.get_xlim())
    under = [y[ok(y) & ((np.log10(n) - x0) / (x1 - x0) >= x_from)] if n is not None else y[ok(y)]
             for n, y in xys]
    top = max((u.max() for u in under if u.size), default=None)
    if top is not None and 0 < top_free < 1:
        if log:
            hi = max(hi, lo * (top / lo) ** (1.0 / (1.0 - top_free)))
        else:
            hi = max(hi, lo + (top - lo) / (1.0 - top_free))
    return lo, hi


def finish_panel(ax, xys, side, yscale, ylim_side, top_free, fontsize, xlabel, label, floor_y):
    """Shared panel cosmetics: y-limits, round log ticks, tick and label sizes, and the in-plot label.
    Returns the axes-fraction height of floor_y, which is what a lower legend has to clear."""
    ax.set_yscale(yscale)
    lo, hi = ylim_side if ylim_side else default_ylim(ax, xys, yscale, top_free)
    ax.set_ylim(lo, hi)
    if yscale == "log":
        vals = log_ticks(lo, hi)
        if vals:
            ax.yaxis.set_major_locator(ticker.FixedLocator(vals))
            ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:g}"))
            ax.yaxis.set_minor_locator(ticker.LogLocator(subs="all"))
            ax.yaxis.set_minor_formatter(ticker.NullFormatter())
    ax.tick_params(which="major", length=6, width=1.0, labelsize=fontsize)
    ax.tick_params(which="minor", length=3.5, width=0.8)
    ax.grid(axis="x", which="major", alpha=0.15, lw=0.8)
    if label:
        ax.text(0.5, 0.965, label, transform=ax.transAxes, ha="center", va="top",
                fontsize=fontsize + 6)
    ax.set_xlabel(xlabel, fontsize=fontsize + 2)
    ax.set_ylabel(f"{SIDE_NAME[side]} Test MSE", fontsize=fontsize + 2)
    return ax.transAxes.inverted().transform(ax.transData.transform((ax.get_xlim()[0], floor_y)))[1]


def place_legend(ax, floor_frac, legend_loc, fontsize):
    """The legend, kept off an oracle line that runs along the bottom of the axes."""
    leg = dict(loc=legend_loc, framealpha=1.0, fontsize=fontsize)
    if legend_loc.startswith("lower") and 0.0 <= floor_frac < 0.2:
        leg["bbox_to_anchor"] = ({"left": 0.0, "center": 0.5, "right": 1.0}[legend_loc.split()[-1]],
                                 floor_frac + 0.01)
    ax.legend(**leg)


def draw_n_panel(ax, runs, T, side, yscale="log", ylim=None, band=True, theory=False,
                 legend_loc="upper right", xlabel="$n$", top_free=0.25, ms=6, fontsize=15, t_label=True,
                 align="dmmse", align_base="first", oracle_band=True, floor3=False):
    """One n-curve panel on `ax`: each run's measured curve, the dMMSE and Ridge oracles, and the
    theory curve if theory=True. Returns the axes-fraction height of the lowest oracle line, for
    place_legend. Arguments as plot_fig2 (ylim is the {side: (lo, hi)} dict)."""
    ylim = ylim or {}
    if not (t_label or legend_loc.startswith("upper")):
        top_free = 0.0                      # nothing sits along the top to keep clear of
    ref = next(recs for _, recs in runs if any(r["T"] == T for r in recs))
    xys = []                                # (n, y) of everything drawn; n=None for the oracle lines
    for k, (label, records) in enumerate(runs):
        offs, _ = pool_offsets(records, T, side, align, align_base)
        n, m, s = curve_stats(records, T, side, band, offs)
        if not len(n):
            continue
        col, mk = STYLES[k % len(STYLES)]
        if np.any(s > 0):
            ax.fill_between(n, m - s, m + s, color=col, alpha=0.2, lw=0, zorder=3)
        ax.plot(n, m, "-", marker=mk, color=col, lw=2.0, ms=ms, label=label, zorder=4)
        xys += [(n, m - s), (n, m + s)]
    roffs, _ = pool_offsets(ref, T, side, align, align_base)
    th = theory_curve(ref, T, side, roffs) if theory else None
    if th is not None:
        ax.plot(*th, "--", color="0.3", lw=2.0, label="Theory", zorder=3.5)
        xys.append(th)
    if floor3 and any("floor3" in r for r in ref):
        # the n -> infinity error of the best degree-3 polynomial on these pools, from the run's records
        y = float(np.mean([r["floor3"][side] - roffs.get(r.get("trial", 0), 0.0)
                           for r in ref if r["T"] == T and "floor3" in r]))
        ax.axhline(y, color="0.45", ls=":", lw=1.6, zorder=2, label=r"Kernel floor deg $\leq$ 3")
        xys.append((None, np.array([y])))
    oys = []
    for key, lab, col, lw in ORACLES:
        y, sd = oracle_stats(ref, T, key, side, roffs, band)
        ax.axhline(y, color=col, lw=lw, label=lab, zorder=2)
        if oracle_band and sd > 0:
            ax.axhspan(y - sd, y + sd, color=col, alpha=0.15, lw=0, zorder=1)
        lo_y = y - sd if oracle_band else y
        xys.append((None, np.array([lo_y, y + sd if oracle_band else y])))
        oys.append(lo_y)
    ax.set_xscale("log")
    return finish_panel(ax, xys, side, yscale, ylim.get(side), top_free, fontsize, xlabel,
                        f"T={T}" if t_label else None, min(oys))


def plot_fig2(runs, T, sides=("ood",), yscale="log", ylim=None, band=True, theory=False,
              legend_loc="upper right", xlabel="$n$", top_free=0.25, ms=6, fontsize=15, t_label=True,
              align="dmmse", align_base="first", oracle_band=True, floor3=False):
    """Measured error against n at task diversity T, one panel per side, figure-2 style.

    runs: [(label, records)]; each run is one coloured series, with a +/- 1 standard deviation band
    across its trials if `band`. The oracle lines -- and the theory curve, if theory=True -- come from
    the first run that has T. align/align_base: put each trial's pool on one baseline level before
    averaging (see pool_offsets); None averages the raw curves. top_free: fraction of the axis height
    kept clear of data under the T label and an upper legend when the y-limits are automatic (0 = fit
    tight). fontsize: tick labels and legend; axis labels are 2pt larger, the T label 6pt.
    """
    fig, axs = plt.subplots(1, len(sides), figsize=(6.0 * len(sides), 4.5), squeeze=False)
    axs = axs[0]
    floors = [draw_n_panel(ax, runs, T, side, yscale, ylim, band, theory, legend_loc, xlabel, top_free,
                           ms, fontsize, t_label, align, align_base, oracle_band, floor3)
              for ax, side in zip(axs, sides)]
    place_legend(axs[0], floors[0], legend_loc, fontsize)
    fig.tight_layout()
    return fig, axs

def value_at(record, n, side, offs=None, tol=2.0):
    """The record's curve value nearest n in log distance, shifted onto the baseline pool.

    nan if the nearest point is further than a factor `tol` from n: a run that stopped early should
    drop out of the read-off rather than quietly contribute its own endpoint instead.
    """
    row = min(record["curve"], key=lambda c: abs(np.log(c["n"]) - np.log(n)))
    if abs(np.log(row["n"]) - np.log(n)) > np.log(tol):
        return np.nan
    return row[side] - (offs or {}).get(record.get("trial", 0), 0.0)


def last_n(records):
    """The largest n on the loaded curves: the default read-off point for the T sweep."""
    return max(c["n"] for r in records for c in r.get("curve", []))


def sweep_stats(records, Ts, side, n_at, band=True, align="dmmse", align_base="first"):
    """(T, mean, sd) of the measured curve read at n_at, one point per T, over trials."""
    out, m, s = [], [], []
    for T in Ts:
        recs = [r for r in records if r["T"] == T and r.get("curve")]
        if not recs:
            continue
        offs, _ = pool_offsets(records, T, side, align, align_base)
        v = np.array([value_at(r, n_at, side, offs) for r in recs], float)
        v = v[np.isfinite(v)]
        if not v.size:
            continue
        out.append(T)
        m.append(v.mean())
        s.append(v.std(ddof=1) if band and v.size > 1 else 0.0)
    return np.array(out, float), np.array(m), np.array(s)


def plot_T_sweep(runs, Ts=None, n_at=None, sides=("ood",), yscale="log", ylim=None, band=True,
                 legend_loc="lower left", xlabel="Task Diversity $T$", top_free=0.25, ms=6,
                 fontsize=15, n_label=True, align="dmmse", align_base="first", oracle_band=True,
                 oracle_markers=None):
    """The same curves read at one n and swept over T, one panel per side, in the same style.

    Ts: which T to show (default every T in the runs). n_at: where each curve is read (default its
    last point); the nearest point in log n is used. The oracles depend on T, so here they are lines
    over T, taken from the first run, on the same shifted pools as the curves.
    """
    ylim = ylim or {}
    ref = runs[0][1]
    Ts = sorted({r["T"] for _, recs in runs for r in recs}) if Ts is None else list(Ts)
    n_at = n_at or last_n(ref)
    if not (n_label or legend_loc.startswith("upper")):
        top_free = 0.0
    floors = []
    fig, axs = plt.subplots(1, len(sides), figsize=(6.0 * len(sides), 4.5), squeeze=False)
    axs = axs[0]
    for ax, side in zip(axs, sides):
        xys = []
        for k, (label, records) in enumerate(runs):
            t, m, sd = sweep_stats(records, Ts, side, n_at, band, align, align_base)
            if not len(t):
                continue
            col, mk = STYLES[k % len(STYLES)]
            if np.any(sd > 0):
                ax.fill_between(t, m - sd, m + sd, color=col, alpha=0.2, lw=0, zorder=3)
            ax.plot(t, m, "-", marker=mk, color=col, lw=2.0, ms=ms, label=label, zorder=4)
            xys += [(t, m - sd), (t, m + sd)]
        oys, ts = [], [T for T in Ts if any(r["T"] == T for r in ref)]
        tf = np.array(ts, float)
        for key, lab, col, lw in ORACLES:
            ms_ = [oracle_stats(ref, T, key, side, pool_offsets(ref, T, side, align, align_base)[0],
                                band) for T in ts]
            y = np.array([a for a, _ in ms_])
            sd = np.array([b for _, b in ms_]) if oracle_band else np.zeros(len(ms_))
            ax.plot(tf, y, "-", color=col, lw=lw, label=lab, zorder=2,
                    marker=(oracle_markers or {}).get(key), ms=ms - 1, mec=col, mfc=col)
            if np.any(sd > 0):
                ax.fill_between(tf, y - sd, y + sd, color=col, alpha=0.15, lw=0, zorder=1)
            xys += [(tf, y - sd), (tf, y + sd)]
            oys += list(y - sd)
        ax.set_xscale("log", base=2)
        ax.xaxis.set_major_locator(ticker.FixedLocator(Ts))
        ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda v, _: f"{v:g}"))
        ax.xaxis.set_minor_locator(ticker.NullLocator())
        floors.append(finish_panel(ax, xys, side, yscale, ylim.get(side), top_free, fontsize, xlabel,
                                   f"n={n_at:,.0f}" if n_label else None, min(oys, default=np.nan)))
    place_legend(axs[0], floors[0], legend_loc, fontsize)
    fig.tight_layout()
    return fig, axs

In [ ]:
FIGDIR = os.path.join(RESULTPATH, "figs")
os.makedirs(FIGDIR, exist_ok=True)

have = sorted({r["T"] for _, recs in runs for r in recs})
for T in CURVE_TS:
    if T not in have:
        print(f"no records at T={T}; the loaded runs have T = {have}")
        continue
    for label, recs in runs:            # what the pool alignment does, before it is hidden in a mean
        offs, base = pool_offsets(recs, T, SIDES[0], ALIGN, ALIGN_BASE)
        if len(offs) > 1:
            print(f"T={T} {label}: {SIDES[0].upper()} baseline dMMSE {base:.3f}; shifts "
                  + ", ".join(f"trial {t} {-d:+.3f}" for t, d in sorted(offs.items())))
    fig, axs = plot_fig2(runs, T, SIDES, yscale=YSCALE, ylim=YLIM, band=SD_BAND, legend_loc=LEGEND_LOC,
                         xlabel=XLABEL, top_free=TOP_FREE, fontsize=FONTSIZE, t_label=T_LABEL,
                         align=ALIGN, align_base=ALIGN_BASE, oracle_band=ORA_BAND)
    stem = os.path.join(FIGDIR, f"rf_transience_fig2_{TAG}_T{T}_{'-'.join(SIDES)}")
    fig.savefig(stem + ".pdf", bbox_inches="tight")
    fig.savefig(stem + ".png", dpi=200, bbox_inches="tight")
    print("saved", stem + ".{pdf,png}")
    plt.show()

In [ ]:
# Where each curve bottoms out and how far it climbs after (needs transience_utils from the clone).
try:
    import transience_utils as TU
    for label, records in runs:
        print(label)
        TU.summary_table(records, sorted({r["T"] for r in records}))
except ImportError as e:
    print(f"transience_utils not importable ({e}); skipping the summary table")

## The same curves against task diversity T

Each curve is read at one n -- the last point by default, `N_AT` for any other -- and plotted against T.
The oracles depend on T, so here they are lines over T rather than horizontal ones.

In [ ]:
N_AT     = None                     # read every curve at this n (None = its last point)
SWEEP_TS = None                     # which T to sweep (None = every T in the loaded runs)
YLIM_T   = {}                       # e.g. {"ood": (0.25, 1.5)}; default fits the curves and oracles
BASE_T   = "mean"                   # baseline pool per T for the sweep: "mean" keeps each T at its own
                                    #   mean level (the shifts cancel, so the oracle lines stay smooth);
                                    #   "first" would let trial 0's pool set the level at every T

N_AT = N_AT or last_n(runs[0][1])
have = sorted({r["T"] for _, recs in runs for r in recs})
if len(have) < 2:
    print(f"only one T ({have}) in these runs -- a sweep needs several")
else:
    fig, axs = plot_T_sweep(runs, SWEEP_TS, N_AT, SIDES, yscale=YSCALE, ylim=YLIM_T, band=SD_BAND,
                            legend_loc=LEGEND_LOC, top_free=TOP_FREE, fontsize=FONTSIZE,
                            n_label=T_LABEL, align=ALIGN, align_base=BASE_T, oracle_band=ORA_BAND)
    stem = os.path.join(FIGDIR, f"rf_transience_fig2_Tsweep_{TAG}_n{int(N_AT)}_{'-'.join(SIDES)}")
    fig.savefig(stem + ".pdf", bbox_inches="tight")
    fig.savefig(stem + ".png", dpi=200, bbox_inches="tight")
    print("saved", stem + ".{pdf,png}")
    plt.show()